# Task Classification

Train classifiers to predict the task labels from `02_task_discovery.ipynb`. The labels come from manually interpreted clusters. Each example uses only the activity observed in its prefix window.


## 0. Configuration

Paths come from `configs/paths.json`. New model files go to the run output directory.


In [ ]:
from __future__ import annotations

from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import json
import math
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from pathlib import Path
import sys

_root = Path.cwd().resolve()
while not (_root / 'project_paths.py').is_file():
    if _root == _root.parent:
        raise RuntimeError('Open this notebook from within the ide-task-inference repository.')
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from project_paths import get_paths, export_discovery_results
paths = get_paths()

paths.ensure_runtime_dirs()
PROJECT_ROOT = paths.root
PHASE2_OUTPUT_DIR = paths.discovery
OUTPUT_DIR = paths.classification_results
TRAINING_INTERMEDIATE_DIR = paths.training
MODEL_DIR = paths.models
FIGURE_DIR = OUTPUT_DIR / "figures"

for directory in [OUTPUT_DIR, MODEL_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEGMENTS_PATH = PHASE2_OUTPUT_DIR / "context_clustered_segments.parquet"
CLUSTER_SUMMARY_PATH = PHASE2_OUTPUT_DIR / "context_cluster_process_summary.parquet"
PIPELINE_SUMMARY_PATH = PHASE2_OUTPUT_DIR / "context_pipeline_summary.json"

MIN_CLUSTER_PROBABILITY = 0.35
MIN_PREFIX_EVENTS = 2
PREFIX_SAMPLES_PER_SEGMENT = 8
MAX_HISTORY_EVENTS = 50
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

PREFER_XGBOOST = True
USE_GPU_IF_AVAILABLE = True
DEFAULT_CONFIDENCE_THRESHOLD = 0.45
# Per-class thresholds are derived on the validation split: for each subtask,
# the smallest threshold whose answered-precision reaches the target below.
PER_CLASS_PRECISION_TARGET = 0.80
THRESHOLD_MIN_SUPPORT = 50
# When the subtask is below its threshold but the summed family probability is
# confident, the runtime API degrades to a family-level answer instead of
# reporting nothing.
FAMILY_FALLBACK_THRESHOLD = 0.60

print("Project root:", PROJECT_ROOT)
print("02 outputs:", PHASE2_OUTPUT_DIR)
print("Model output:", MODEL_DIR)


## 1. Load the task taxonomy

Read the cluster-to-task mapping from `task_taxonomy.json`. The fallback uses `pipeline/process_mining/task_taxonomy.py`.


In [ ]:
import sys
from dataclasses import dataclass

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.process_mining.task_taxonomy import load_taxonomy, taxonomy_payload

# Single source of truth: the taxonomy exported by notebook 02
# (pipeline/process_mining/task_taxonomy.py -> data/intermediate/discovery/task_taxonomy.json).
TAXONOMY_PATH = PHASE2_OUTPUT_DIR / "task_taxonomy.json"
if TAXONOMY_PATH.exists():
    TAXONOMY = load_taxonomy(TAXONOMY_PATH)
    print("Loaded taxonomy export:", TAXONOMY_PATH)
else:
    TAXONOMY = taxonomy_payload()
    print(
        "task_taxonomy.json not found in 02 outputs; using pipeline/process_mining/task_taxonomy.py directly. "
        "Re-run the 02 export cell to regenerate the export."
    )

print("Taxonomy schema:", TAXONOMY["schema_version"], "| run:", TAXONOMY["run_id"])
if TAXONOMY.get("unmapped_cluster_ids"):
    print("WARNING: 02 reported non-noise clusters missing from the taxonomy map:", TAXONOMY["unmapped_cluster_ids"])


@dataclass(frozen=True)
class TaskLabel:
    task_family: str
    subtask: str
    description: str


SUBTASK_INFO = {item["subtask"]: item for item in TAXONOMY["subtasks"]}
CLUSTER_TO_TASK: dict[int, TaskLabel] = {
    int(cluster_id): TaskLabel(
        SUBTASK_INFO[subtask]["task_family"],
        subtask,
        SUBTASK_INFO[subtask]["description"],
    )
    for cluster_id, subtask in TAXONOMY["cluster_to_subtask"].items()
}

TASK_FAMILY_ORDER = list(TAXONOMY["task_families"])
SUBTASK_ORDER = [item["subtask"] for item in TAXONOMY["subtasks"]]

TASK_LABEL_TABLE = pd.DataFrame(
    [
        {
            "cluster_id": cluster_id,
            "task_family": label.task_family,
            "subtask": label.subtask,
            "description": label.description,
        }
        for cluster_id, label in sorted(CLUSTER_TO_TASK.items())
    ]
)

TASK_LABEL_TABLE.head(30)

### 1.1 Workflow overview

The diagram connects cluster interpretation, classifier training, and inference from recent activity.


In [ ]:
# Reporting figure 1.1: end-to-end architecture - offline task discovery (02) feeding real-time inference (03).
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

arch_summary = json.loads(PIPELINE_SUMMARY_PATH.read_text(encoding="utf-8")) if PIPELINE_SUMMARY_PATH.exists() else {}
arch_vectorizer_path = MODEL_DIR / "feature_vectorizer.joblib"
arch_feature_total = len(joblib.load(arch_vectorizer_path).feature_names_) if arch_vectorizer_path.exists() else None


def arch_fmt(arch_value):
    arch_value = int(arch_value)
    if arch_value >= 1_000_000:
        return f"{arch_value / 1_000_000:.1f}M"
    if arch_value >= 10_000:
        return f"{arch_value / 1_000:.0f}K"
    return f"{arch_value:,}"


def arch_line(arch_title, arch_detail):
    return f"{arch_title}\n({arch_detail})" if arch_detail else arch_title


ARCH_STYLES = {
    "data": ("#EAF1F8", "#4C78A8"),
    "model": ("#FDEEDC", "#B15C1E"),
    "product": ("#E8F2E6", "#3E7A38"),
}


def arch_box(arch_ax, arch_x, arch_y, arch_w, arch_h, arch_text, arch_kind):
    arch_face, arch_edge = ARCH_STYLES[arch_kind]
    arch_ax.add_patch(
        FancyBboxPatch(
            (arch_x, arch_y), arch_w, arch_h,
            boxstyle="round,pad=0.02,rounding_size=0.10",
            facecolor=arch_face, edgecolor=arch_edge, linewidth=1.3, zorder=3,
        )
    )
    arch_ax.text(arch_x + arch_w / 2, arch_y + arch_h / 2, arch_text,
                 ha="center", va="center", fontsize=8.2, zorder=4)
    return (arch_x, arch_y, arch_w, arch_h)


def arch_arrow(arch_ax, arch_from, arch_to, arch_label=None, arch_rad=0.0):
    arch_ax.add_patch(
        FancyArrowPatch(
            arch_from, arch_to,
            connectionstyle=f"arc3,rad={arch_rad}",
            arrowstyle="-|>", mutation_scale=12, color="#5A5A5A", linewidth=1.15, zorder=2,
        )
    )
    if arch_label:
        arch_mx, arch_my = (arch_from[0] + arch_to[0]) / 2, (arch_from[1] + arch_to[1]) / 2
        arch_ax.text(arch_mx + 0.12, arch_my, arch_label, ha="left", va="center",
                     fontsize=7.4, color="#5A5A5A", style="italic", zorder=4)


fig, ax = plt.subplots(figsize=(15, 7.6))

ax.add_patch(FancyBboxPatch((0.25, 5.35), 19.5, 4.25, boxstyle="round,pad=0.02,rounding_size=0.14",
                            facecolor="#4C78A8", alpha=0.06, edgecolor="#4C78A8", linewidth=1.0, zorder=0))
ax.text(0.55, 9.28, "Offline — task discovery (notebook 02)", fontsize=10.5, fontweight="bold", color="#2F4B6E")
ax.add_patch(FancyBboxPatch((0.25, 0.35), 19.5, 3.45, boxstyle="round,pad=0.02,rounding_size=0.14",
                            facecolor="#F58518", alpha=0.06, edgecolor="#B15C1E", linewidth=1.0, zorder=0))
ax.text(0.55, 3.42, "Online — real-time inference (notebook 03 → runtime)", fontsize=10.5, fontweight="bold", color="#7A4712")

arch_offline = [
    ("data", arch_line("IDE event logs", f"{arch_summary['loaded_sessions']:,} sessions" if "loaded_sessions" in arch_summary else None)),
    ("data", arch_line("Canonical\nevents", arch_fmt(arch_summary["canonical_events"]) if "canonical_events" in arch_summary else None)),
    ("data", arch_line("Activity\nbursts", arch_fmt(arch_summary["activity_bursts"]) if "activity_bursts" in arch_summary else None)),
    ("model", arch_line("Representation-shift\nsegmentation", f"{arch_fmt(arch_summary['segments'])} segments" if "segments" in arch_summary else None)),
    ("model", arch_line("HDBSCAN\nclustering", f"{arch_summary['clusters_excluding_noise']} clusters" if "clusters_excluding_noise" in arch_summary else None)),
    ("product", arch_line("Manual task\ntaxonomy", f"{len(TAXONOMY['subtasks'])} subtasks / {len(TAXONOMY['task_families'])} families")),
]
arch_w, arch_h = 2.65, 1.5
arch_xs = [0.62 + arch_i * 3.18 for arch_i in range(6)]
arch_boxes = [arch_box(ax, arch_x, 7.15, arch_w, arch_h, arch_text, arch_kind)
              for arch_x, (arch_kind, arch_text) in zip(arch_xs, arch_offline)]
for arch_left, arch_right in zip(arch_boxes, arch_boxes[1:]):
    arch_arrow(ax, (arch_left[0] + arch_w, 7.15 + arch_h / 2), (arch_right[0], 7.15 + arch_h / 2))

arch_fuzzy = arch_box(ax, arch_xs[4] - 1.0, 5.62, arch_w, 1.05, "Fuzzy process mining\n(per-cluster evidence)", "model")
arch_arrow(ax, (arch_xs[4] + arch_w / 2 - 0.6, 7.15), (arch_fuzzy[0] + arch_w * 0.45, 6.67))
arch_arrow(ax, (arch_fuzzy[0] + arch_w, 6.14), (arch_xs[5] + arch_w * 0.45, 7.15))

arch_pseudo = arch_box(ax, arch_xs[5], 4.28, arch_w, 0.92, "Pseudo-labelled segments", "data")
arch_arrow(ax, (arch_xs[5] + arch_w / 2, 7.15), (arch_xs[5] + arch_w / 2, 5.20))

arch_online = [
    ("data", "Live IDE\nevents"),
    ("data", arch_line("Sliding prefix\nwindow", f"≤{MAX_HISTORY_EVENTS} events")),
    ("data", arch_line("Window\nfeatures", f"{arch_feature_total}-dim" if arch_feature_total else None)),
    ("model", "XGBoost heads\n(subtask + family)"),
    ("model", "Per-class confidence\nthresholds"),
    ("data", "Task inference\n(subtask, family,\nconfidence)"),
    ("product", "SmartContextAdvisor\nplugin"),
]
arch_w2 = 2.42
arch_xs2 = [0.62 + arch_i * 2.73 for arch_i in range(7)]
arch_boxes2 = [arch_box(ax, arch_x, 1.35, arch_w2, arch_h, arch_text, arch_kind)
               for arch_x, (arch_kind, arch_text) in zip(arch_xs2, arch_online)]
for arch_left, arch_right in zip(arch_boxes2, arch_boxes2[1:]):
    arch_arrow(ax, (arch_left[0] + arch_w2, 1.35 + arch_h / 2), (arch_right[0], 1.35 + arch_h / 2))

arch_arrow(ax, (arch_xs[5] + arch_w / 2, 4.28), (arch_xs2[3] + arch_w2 / 2, 2.85),
           arch_label="training supervision", arch_rad=0.12)

ax.set_xlim(0, 20)
ax.set_ylim(0, 10)
ax.set_axis_off()
ax.set_title("Task-inference system: offline discovery pipeline and real-time classification path", fontsize=13, pad=8)
fig.tight_layout()
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_DIR / "report_architecture_overview.png", dpi=160, bbox_inches="tight")
plt.show()

## 2. Assign segment labels

Keep non-noise segments with a cluster membership probability of at least 0.35. Each retained segment inherits its cluster's subtask and task family.


In [ ]:
def load_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


for required_path in [SEGMENTS_PATH, CLUSTER_SUMMARY_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required 02 output: {required_path}")

segments = pd.read_parquet(SEGMENTS_PATH)
cluster_summary = pd.read_parquet(CLUSTER_SUMMARY_PATH)
pipeline_summary = load_json(PIPELINE_SUMMARY_PATH)

non_noise_clusters = sorted(int(c) for c in segments.loc[~segments["is_noise"], "cluster_id"].dropna().unique())
missing_clusters = sorted(set(non_noise_clusters) - set(CLUSTER_TO_TASK))
extra_clusters = sorted(set(CLUSTER_TO_TASK) - set(non_noise_clusters))

if missing_clusters:
    raise ValueError(f"The manual taxonomy is missing non-noise clusters from 02: {missing_clusters}")
if extra_clusters:
    print("Warning: manual taxonomy includes clusters not present in this run:", extra_clusters)

labeled_segments = segments.loc[
    (~segments["is_noise"]) &
    (segments["cluster_probability"].fillna(0.0) >= MIN_CLUSTER_PROBABILITY) &
    (segments["cluster_id"].isin(CLUSTER_TO_TASK))
].copy()

labeled_segments["task_family"] = labeled_segments["cluster_id"].map(lambda cid: CLUSTER_TO_TASK[int(cid)].task_family)
labeled_segments["subtask"] = labeled_segments["cluster_id"].map(lambda cid: CLUSTER_TO_TASK[int(cid)].subtask)
labeled_segments["task_label"] = labeled_segments["task_family"] + " :: " + labeled_segments["subtask"]

segment_label_summary = (
    labeled_segments.groupby(["task_family", "subtask"], observed=True)
    .agg(
        segments=("segment_id", "count"),
        clusters=("cluster_id", "nunique"),
        mean_cluster_probability=("cluster_probability", "mean"),
        median_activity_count=("activity_count", "median"),
        median_duration_seconds=("duration_seconds", "median"),
    )
    .reset_index()
    .sort_values(["task_family", "subtask"])
)

print("02 loaded sessions:", pipeline_summary.get("loaded_sessions"))
print("02 segments:", len(segments))
print("Labeled non-noise segments used for training:", len(labeled_segments))
print("Labels:", labeled_segments["task_label"].nunique())
segment_label_summary


## 3. Activity fields for classification

Read the activity sequence and its ordered context, including file role, command category, navigation kind, and debugger or version control actions. Older exports without `context_trace` use activity tokens alone.


In [ ]:
# Same activity->family map 02 clusters with (process_mining/case_segmentation.py).
# The bridge cell already put PROJECT_ROOT on sys.path.
from pipeline.process_mining.case_segmentation import ACTIVITY_FAMILY as ACTIVITY_TO_FAMILY

NUMERIC_CONTEXT_COLUMNS = [
    "activity_count",
    "event_count",
    "duration_seconds",
    "file_count",
    "unique_file_count",
    "unique_symbol_count",
    "file_context_entropy",
    "command_context_entropy",
    "close_boundary_distance",
]

CATEGORICAL_CONTEXT_COLUMNS = [
    "dominant_family",
    "dominant_activity",
    "start_reason",
    "close_reason",
]


def parse_trace(trace: Any) -> list[str]:
    """Parse the compact trace string exported by 02 into activity tokens."""
    if trace is None or (isinstance(trace, float) and np.isnan(trace)):
        return []
    if isinstance(trace, list):
        return [str(x).strip() for x in trace if str(x).strip()]
    return [part.strip() for part in str(trace).split(" -> ") if part.strip()]


def infer_event_family(activity: str | None) -> str:
    if not activity:
        return "unknown"
    return ACTIVITY_TO_FAMILY.get(str(activity), "other")


def normalize_intermediate_event(event: dict[str, Any] | str) -> dict[str, Any]:
    """Normalize dataset-derived or plugin-derived events into the training protocol."""
    if isinstance(event, str):
        activity = event
        return {"activity": activity, "action_type": activity, "family": infer_event_family(activity)}

    activity = event.get("activity") or event.get("action_type") or event.get("event_type") or event.get("type") or "UNKNOWN"
    normalized = dict(event)
    normalized["activity"] = str(activity)
    normalized["action_type"] = str(event.get("action_type") or activity)
    normalized["family"] = str(event.get("family") or infer_event_family(str(activity)))
    return normalized


def trace_to_intermediate_events(trace: Any) -> list[dict[str, Any]]:
    """Convert a 02 segment trace into intermediate events for training."""
    return [normalize_intermediate_event(activity) for activity in parse_trace(trace)]


def parse_context_trace(value: Any) -> list[dict[str, Any]]:
    """Parse the ordered per-activity context JSON exported by 02 (may be absent)."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    if isinstance(value, list):
        return [dict(item) for item in value if isinstance(item, dict)]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = json.loads(text)
    except (TypeError, ValueError):
        return []
    if not isinstance(parsed, list):
        return []
    return [dict(item) for item in parsed if isinstance(item, dict)]


def segment_to_intermediate_events(row_dict: dict[str, Any]) -> list[dict[str, Any]]:
    """Build context-carrying intermediate events for one 02 segment row."""
    tokens = parse_trace(row_dict.get("trace"))
    contexts = parse_context_trace(row_dict.get("context_trace"))
    if contexts and len(contexts) == len(tokens):
        return [
            normalize_intermediate_event({**context, "activity": token})
            for token, context in zip(tokens, contexts, strict=False)
        ]
    # Fallback for artifacts produced before 02 exported context traces.
    return trace_to_intermediate_events(row_dict.get("trace"))


def safe_float(value: Any, default: float = 0.0) -> float:
    try:
        value = float(value)
    except (TypeError, ValueError):
        return default
    if not math.isfinite(value):
        return default
    return value


def log1p_feature(value: Any) -> float:
    return float(np.log1p(max(0.0, safe_float(value))))


In [ ]:
def extract_window_features(
    events: list[dict[str, Any] | str],
    segment_context: dict[str, Any] | None = None,
    max_history_events: int = MAX_HISTORY_EVENTS,
) -> dict[str, float]:
    """Extract deployment-friendly features from a rolling intermediate event window.

    The function favors counts, recent order, and lightweight context. These features can be
    reconstructed online by SmartContextAdvisor without knowing the original dataset schema.
    """
    normalized = [normalize_intermediate_event(event) for event in events]
    if max_history_events:
        normalized = normalized[-max_history_events:]

    activities = [event["activity"] for event in normalized]
    families = [event.get("family", infer_event_family(event.get("activity"))) for event in normalized]
    features: dict[str, float] = {}

    n = len(activities)
    features["num:event_count"] = float(n)
    features["num:log_event_count"] = float(np.log1p(n))
    features["num:unique_activity_count"] = float(len(set(activities)))
    features["num:unique_family_count"] = float(len(set(families)))

    if activities:
        features[f"first_activity={activities[0]}"] = 1.0
        features[f"last_activity={activities[-1]}"] = 1.0
        features[f"last_family={families[-1]}"] = 1.0

    activity_counts = Counter(activities)
    family_counts = Counter(families)
    for activity, count in activity_counts.items():
        features[f"activity_count={activity}"] = float(count)
        features[f"activity_ratio={activity}"] = float(count / max(1, n))
    for family, count in family_counts.items():
        features[f"family_count={family}"] = float(count)
        features[f"family_ratio={family}"] = float(count / max(1, n))

    # file_role is the defining signal for test-code work, so it gets
    # window-wide ratios in addition to the recency features below.
    role_counts = Counter(
        str(event.get("file_role"))
        for event in normalized
        if event.get("file_role") not in [None, "", "unknown"]
    )
    for role, count in role_counts.items():
        features[f"file_role_count={role}"] = float(count)
        features[f"file_role_ratio={role}"] = float(count / max(1, n))

    recent = activities[-8:]
    for activity, count in Counter(recent).items():
        features[f"recent_activity_count={activity}"] = float(count)

    for left, right in zip(activities, activities[1:], strict=False):
        key = f"bigram={left}->{right}"
        features[key] = features.get(key, 0.0) + 1.0
    for a, b, c in zip(activities, activities[1:], activities[2:], strict=False):
        key = f"trigram={a}->{b}->{c}"
        features[key] = features.get(key, 0.0) + 1.0

    for event in normalized[-10:]:
        for field in ["command_category", "navigation_kind", "debug_action", "vcs_action", "file_role", "source_event", "source_action"]:
            value = event.get(field)
            if value not in [None, "", "unknown"]:
                key = f"recent_{field}={value}"
                features[key] = features.get(key, 0.0) + 1.0

    if segment_context:
        for column in NUMERIC_CONTEXT_COLUMNS:
            if column in segment_context:
                value = safe_float(segment_context.get(column))
                features[f"ctx:{column}"] = value
                features[f"ctx:log_{column}"] = log1p_feature(value)
        for column in CATEGORICAL_CONTEXT_COLUMNS:
            value = segment_context.get(column)
            if value not in [None, "", "nan"]:
                features[f"ctx:{column}={value}"] = 1.0

    return features


extract_window_features([
    {"activity": "NAVIGATION_BURST", "navigation_kind": "go_to_definition"},
    {"activity": "IDE_COMMAND_ACTIVITY", "command_category": "other"},
    {"activity": "EDIT_BURST", "file_role": "source"},
])


## 4. Build prefix examples

Sample several prefixes from each labelled segment. Features use up to the 50 most recent activity items in each prefix, and every prefix inherits its segment's label.


In [ ]:
def sampled_prefix_lengths(trace_length: int, min_prefix_events: int, samples_per_segment: int) -> list[int]:
    if trace_length <= 0:
        return []
    start = min(min_prefix_events, trace_length)
    if trace_length <= samples_per_segment + start:
        return list(range(start, trace_length + 1))
    values = np.linspace(start, trace_length, num=samples_per_segment)
    return sorted(set(int(round(v)) for v in values if 1 <= int(round(v)) <= trace_length))


def build_training_examples(segments_frame: pd.DataFrame) -> tuple[list[dict[str, float]], pd.DataFrame]:
    feature_rows: list[dict[str, float]] = []
    metadata_rows: list[dict[str, Any]] = []

    for row in segments_frame.itertuples(index=False):
        row_dict = row._asdict()
        events = segment_to_intermediate_events(row_dict)
        if not events:
            continue

        prefix_lengths = sampled_prefix_lengths(len(events), MIN_PREFIX_EVENTS, PREFIX_SAMPLES_PER_SEGMENT)
        for prefix_len in prefix_lengths:
            window_events = events[:prefix_len][-MAX_HISTORY_EVENTS:]
            # No segment_context: full-segment stats (final duration/counts, dominant
            # activity, close_reason) do not exist mid-task at runtime and leaked the
            # pseudo-label, inflating every metric. Window-only features keep the
            # training and deployment feature distributions identical.
            feature_rows.append(extract_window_features(window_events))
            metadata_rows.append(
                {
                    "segment_id": row_dict["segment_id"],
                    "session_index": int(row_dict["session_index"]),
                    "cluster_id": int(row_dict["cluster_id"]),
                    "cluster_probability": float(row_dict["cluster_probability"]),
                    "prefix_len": int(prefix_len),
                    "full_trace_len": int(len(events)),
                    "task_family": row_dict["task_family"],
                    "subtask": row_dict["subtask"],
                    "task_label": row_dict["task_label"],
                }
            )

    return feature_rows, pd.DataFrame(metadata_rows)


feature_dicts, example_meta = build_training_examples(labeled_segments)
if not feature_dicts:
    raise RuntimeError("No training examples were generated from the labeled segments.")

example_meta["sample_weight"] = example_meta["cluster_probability"].clip(0.10, 1.0)

print("Generated realtime-style window examples:", len(example_meta))
print("Unique sessions:", example_meta["session_index"].nunique())
print("Unique task labels:", example_meta["task_label"].nunique())
example_meta.groupby(["task_family", "subtask"], observed=True).agg(
    examples=("segment_id", "count"),
    source_segments=("segment_id", "nunique"),
    sessions=("session_index", "nunique"),
    median_prefix_len=("prefix_len", "median"),
).reset_index().sort_values(["task_family", "subtask"])


## 5. Split the examples

Try session-level splits that cover all labels. The fallback keeps all prefixes from the same segment in one partition.


In [ ]:
def label_coverage(indices: np.ndarray, labels: np.ndarray) -> set[str]:
    return set(labels[indices])


def make_grouped_splits(meta: pd.DataFrame, label_column: str = "task_label") -> tuple[np.ndarray, np.ndarray, np.ndarray, str]:
    labels = meta[label_column].to_numpy()
    groups = meta["session_index"].to_numpy()
    all_labels = set(labels)
    indices = np.arange(len(meta))

    for seed in range(RANDOM_SEED, RANDOM_SEED + 40):
        outer = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
        train_val_idx, test_idx = next(outer.split(indices, labels, groups))
        if label_coverage(test_idx, labels) != all_labels:
            continue

        inner_groups = groups[train_val_idx]
        inner_labels = labels[train_val_idx]
        inner = GroupShuffleSplit(n_splits=1, test_size=VALIDATION_SIZE, random_state=seed + 1000)
        train_rel, val_rel = next(inner.split(train_val_idx, inner_labels, inner_groups))
        train_idx = train_val_idx[train_rel]
        val_idx = train_val_idx[val_rel]
        if label_coverage(train_idx, labels) == all_labels and label_coverage(val_idx, labels) == all_labels:
            return train_idx, val_idx, test_idx, "grouped_by_session"

    # Some labels may be concentrated in too few sessions for a strict grouped-by-session split.
    # The fallback is stratified at the segment level, so different prefix windows from the same
    # adaptive segment still stay in the same split. This avoids the most direct leakage mode.
    segment_labels = meta[["segment_id", label_column]].drop_duplicates("segment_id").reset_index(drop=True)
    train_val_segments, test_segments = train_test_split(
        segment_labels["segment_id"].to_numpy(),
        test_size=TEST_SIZE,
        random_state=RANDOM_SEED,
        stratify=segment_labels[label_column].to_numpy(),
    )
    train_val_frame = segment_labels[segment_labels["segment_id"].isin(train_val_segments)].copy()
    train_segments, val_segments = train_test_split(
        train_val_frame["segment_id"].to_numpy(),
        test_size=VALIDATION_SIZE,
        random_state=RANDOM_SEED + 1,
        stratify=train_val_frame[label_column].to_numpy(),
    )

    train_idx = meta.index[meta["segment_id"].isin(train_segments)].to_numpy()
    val_idx = meta.index[meta["segment_id"].isin(val_segments)].to_numpy()
    test_idx = meta.index[meta["segment_id"].isin(test_segments)].to_numpy()
    return np.asarray(train_idx), np.asarray(val_idx), np.asarray(test_idx), "stratified_by_segment_fallback"


train_idx, val_idx, test_idx, split_strategy = make_grouped_splits(example_meta)
print("Split strategy:", split_strategy)
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "examples": [len(train_idx), len(val_idx), len(test_idx)],
        "sessions": [
            example_meta.iloc[train_idx]["session_index"].nunique(),
            example_meta.iloc[val_idx]["session_index"].nunique(),
            example_meta.iloc[test_idx]["session_index"].nunique(),
        ],
        "labels": [
            example_meta.iloc[train_idx]["task_label"].nunique(),
            example_meta.iloc[val_idx]["task_label"].nunique(),
            example_meta.iloc[test_idx]["task_label"].nunique(),
        ],
    }
)
split_summary


## 6. Train the classifiers

Fit the feature vocabulary on all prefix examples. Train separate models for subtasks and task families, with weights that account for class frequency and cluster membership.


In [ ]:
vectorizer = DictVectorizer(sparse=True)
X_all = vectorizer.fit_transform(feature_dicts)
feature_names = vectorizer.get_feature_names_out().tolist()

print("Vectorized feature matrix:", X_all.shape)
print("First 20 features:", feature_names[:20])

subtask_encoder = LabelEncoder()
family_encoder = LabelEncoder()
subtask_encoder.fit([label for label in SUBTASK_ORDER if label in set(example_meta["subtask"])])
family_encoder.fit([label for label in TASK_FAMILY_ORDER if label in set(example_meta["task_family"])])

y_subtask = subtask_encoder.transform(example_meta["subtask"])
y_family = family_encoder.transform(example_meta["task_family"])
sample_weight = example_meta["sample_weight"].to_numpy(dtype=float)

X_train = X_all[train_idx]
X_val = X_all[val_idx]
X_test = X_all[test_idx]

weights_train = sample_weight[train_idx]
weights_val = sample_weight[val_idx]

print("Subtask classes:", list(subtask_encoder.classes_))
print("Family classes:", list(family_encoder.classes_))


In [ ]:
def try_import_xgboost() -> Any | None:
    try:
        import xgboost as xgb  # type: ignore
        return xgb
    except Exception as exc:
        print("XGBoost is not available; using sklearn fallback.", repr(exc))
        return None


def top_k_accuracy(y_true: np.ndarray, proba: np.ndarray, k: int = 3) -> float:
    if proba.size == 0:
        return float("nan")
    top = np.argsort(proba, axis=1)[:, -k:]
    return float(np.mean([truth in row for truth, row in zip(y_true, top, strict=False)]))


def train_logistic_classifier(name: str, y_train: np.ndarray, y_val: np.ndarray, classes: np.ndarray, fit_weights: np.ndarray) -> dict[str, Any]:
    candidates = [0.25, 0.75, 1.5, 3.0, 6.0]
    results: list[dict[str, Any]] = []
    best: dict[str, Any] | None = None

    for C in candidates:
        model = LogisticRegression(
            C=C,
            max_iter=1200,
            solver="lbfgs",
            random_state=RANDOM_SEED,
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            # Class balancing is already folded into fit_weights.
            model.fit(X_train, y_train, sample_weight=fit_weights)
        val_pred = model.predict(X_val)
        val_proba = model.predict_proba(X_val)
        score = f1_score(y_val, val_pred, average="macro", zero_division=0)
        row = {
            "model_family": "sklearn_logistic_regression",
            "name": name,
            "C": C,
            "validation_macro_f1": float(score),
            "validation_accuracy": float(accuracy_score(y_val, val_pred)),
            "validation_top3_accuracy": top_k_accuracy(y_val, val_proba, k=min(3, len(classes))),
            "estimator": model,
        }
        results.append(row)
        if best is None or row["validation_macro_f1"] > best["validation_macro_f1"]:
            best = row

    assert best is not None
    return {"best": best, "candidates": results}


In [ ]:
def train_xgboost_classifier(name: str, y_train: np.ndarray, y_val: np.ndarray, classes: np.ndarray, fit_weights: np.ndarray) -> dict[str, Any] | None:
    xgb = try_import_xgboost() if PREFER_XGBOOST else None
    if xgb is None:
        return None

    device = "cuda" if USE_GPU_IF_AVAILABLE else "cpu"
    candidates = [
        {"max_depth": 4, "learning_rate": 0.08, "n_estimators": 220, "subsample": 0.90, "colsample_bytree": 0.90},
        {"max_depth": 5, "learning_rate": 0.06, "n_estimators": 300, "subsample": 0.90, "colsample_bytree": 0.85},
        {"max_depth": 6, "learning_rate": 0.04, "n_estimators": 360, "subsample": 0.85, "colsample_bytree": 0.85},
    ]
    results: list[dict[str, Any]] = []
    best: dict[str, Any] | None = None

    for params in candidates:
        model = xgb.XGBClassifier(
            objective="multi:softprob",
            num_class=len(classes),
            eval_metric="mlogloss",
            tree_method="hist",
            device=device,
            random_state=RANDOM_SEED,
            n_jobs=4,
            **params,
        )
        try:
            model.fit(X_train, y_train, sample_weight=fit_weights, verbose=False)
        except Exception as exc:
            if device == "cuda":
                print(f"{name}: CUDA XGBoost failed for {params}; retrying on CPU. Error: {exc!r}")
                model = xgb.XGBClassifier(
                    objective="multi:softprob",
                    num_class=len(classes),
                    eval_metric="mlogloss",
                    tree_method="hist",
                    device="cpu",
                    random_state=RANDOM_SEED,
                    n_jobs=4,
                    **params,
                )
                model.fit(X_train, y_train, sample_weight=fit_weights, verbose=False)
            else:
                raise

        val_pred = model.predict(X_val)
        val_proba = model.predict_proba(X_val)
        score = f1_score(y_val, val_pred, average="macro", zero_division=0)
        row = {
            "model_family": "xgboost",
            "name": name,
            "validation_macro_f1": float(score),
            "validation_accuracy": float(accuracy_score(y_val, val_pred)),
            "validation_top3_accuracy": top_k_accuracy(y_val, val_proba, k=min(3, len(classes))),
            "params": params,
            "device_requested": device,
            "estimator": model,
        }
        results.append(row)
        if best is None or row["validation_macro_f1"] > best["validation_macro_f1"]:
            best = row

    assert best is not None
    return {"best": best, "candidates": results}


def train_best_classifier(name: str, y_train: np.ndarray, y_val: np.ndarray, classes: np.ndarray, fit_weights: np.ndarray) -> dict[str, Any]:
    xgb_result = train_xgboost_classifier(name, y_train, y_val, classes, fit_weights)
    if xgb_result is not None:
        return xgb_result
    return train_logistic_classifier(name, y_train, y_val, classes, fit_weights)


In [ ]:
def balanced_fit_weights(y_train_labels: np.ndarray) -> np.ndarray:
    """Combine cluster-probability weights with inverse class-frequency balancing."""
    counts = np.bincount(y_train_labels)
    class_weight = len(y_train_labels) / (np.count_nonzero(counts) * np.maximum(counts, 1))
    weights = weights_train * class_weight[y_train_labels]
    return weights * (len(weights) / weights.sum())


subtask_training = train_best_classifier("subtask", y_subtask[train_idx], y_subtask[val_idx], subtask_encoder.classes_, balanced_fit_weights(y_subtask[train_idx]))
family_training = train_best_classifier("task_family", y_family[train_idx], y_family[val_idx], family_encoder.classes_, balanced_fit_weights(y_family[train_idx]))

subtask_model = subtask_training["best"]["estimator"]
family_model = family_training["best"]["estimator"]

search_rows = []
for training_result in [subtask_training, family_training]:
    for candidate in training_result["candidates"]:
        row = {k: v for k, v in candidate.items() if k != "estimator"}
        if "params" in row:
            row.update({f"param:{k}": v for k, v in row.pop("params").items()})
        search_rows.append(row)

search_results = pd.DataFrame(search_rows)
search_results.to_csv(OUTPUT_DIR / "classifier_parameter_search.csv", index=False)
search_results


## 7. Evaluate on the test set

Compare predictions with the labels derived from the manual cluster interpretation. The prefixes start at segment boundaries identified during discovery.


In [ ]:
def evaluate_classifier(
    name: str,
    model: Any,
    encoder: LabelEncoder,
    X_split: Any,
    y_true: np.ndarray,
    meta_split: pd.DataFrame,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    y_pred = model.predict(X_split)
    proba = model.predict_proba(X_split) if hasattr(model, "predict_proba") else np.zeros((len(y_true), len(encoder.classes_)))
    confidence = proba.max(axis=1) if proba.size else np.full(len(y_true), np.nan)

    labels = np.arange(len(encoder.classes_))
    report = classification_report(
        y_true,
        y_pred,
        labels=labels,
        target_names=encoder.classes_,
        output_dict=True,
        zero_division=0,
    )
    report_frame = pd.DataFrame(report).T
    report_frame.to_csv(OUTPUT_DIR / f"{name}_classification_report.csv")

    metrics = {
        "name": name,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "top3_accuracy": top_k_accuracy(y_true, proba, k=min(3, len(encoder.classes_))),
        "mean_confidence": float(np.nanmean(confidence)),
    }

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig_width = max(8, 0.55 * len(encoder.classes_))
    fig_height = max(6, 0.50 * len(encoder.classes_))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    cm_display = ConfusionMatrixDisplay(cm, display_labels=encoder.classes_)
    cm_display.plot(ax=ax, xticks_rotation=45, colorbar=False, values_format="d")
    ax.set_title(f"{name} confusion matrix")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{name}_confusion_matrix.png", dpi=160)
    display(fig)
    plt.close(fig)

    pred_frame = meta_split.copy()
    pred_frame[f"{name}_true"] = encoder.inverse_transform(y_true)
    pred_frame[f"{name}_pred"] = encoder.inverse_transform(y_pred)
    pred_frame[f"{name}_confidence"] = confidence
    pred_frame[f"{name}_correct"] = y_true == y_pred
    pred_frame.to_parquet(TRAINING_INTERMEDIATE_DIR / f"{name}_test_predictions.parquet", index=False)

    return report_frame, metrics


subtask_report, subtask_metrics = evaluate_classifier(
    "subtask",
    subtask_model,
    subtask_encoder,
    X_test,
    y_subtask[test_idx],
    example_meta.iloc[test_idx].reset_index(drop=True),
)
family_report, family_metrics = evaluate_classifier(
    "task_family",
    family_model,
    family_encoder,
    X_test,
    y_family[test_idx],
    example_meta.iloc[test_idx].reset_index(drop=True),
)

metrics_frame = pd.DataFrame([subtask_metrics, family_metrics])
metrics_frame.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
metrics_frame


In [ ]:
def prediction_confidence_frame(model: Any, encoder: LabelEncoder, X_split: Any, y_true: np.ndarray, meta_split: pd.DataFrame) -> pd.DataFrame:
    proba = model.predict_proba(X_split)
    pred = np.argmax(proba, axis=1)
    return pd.DataFrame(
        {
            "prefix_len": meta_split["prefix_len"].to_numpy(),
            "full_trace_len": meta_split["full_trace_len"].to_numpy(),
            "task_label": meta_split["task_label"].to_numpy(),
            "true_subtask": encoder.inverse_transform(y_true),
            "pred_subtask": encoder.inverse_transform(pred),
            "confidence": proba.max(axis=1),
            "correct": pred == y_true,
        }
    )


prefix_eval = prediction_confidence_frame(
    subtask_model,
    subtask_encoder,
    X_test,
    y_subtask[test_idx],
    example_meta.iloc[test_idx].reset_index(drop=True),
)
prefix_eval["prefix_bin"] = pd.cut(
    prefix_eval["prefix_len"],
    bins=[0, 3, 5, 10, 20, 50, 10000],
    labels=["1-3", "4-5", "6-10", "11-20", "21-50", "51+"],
    include_lowest=True,
)
prefix_curve = prefix_eval.groupby("prefix_bin", observed=True).agg(
    examples=("correct", "size"),
    accuracy=("correct", "mean"),
    mean_confidence=("confidence", "mean"),
).reset_index()
prefix_curve.to_csv(OUTPUT_DIR / "prefix_length_performance.csv", index=False)

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(prefix_curve["prefix_bin"].astype(str), prefix_curve["accuracy"], marker="o", label="accuracy")
ax1.plot(prefix_curve["prefix_bin"].astype(str), prefix_curve["mean_confidence"], marker="s", label="mean confidence")
ax1.set_ylim(0, 1.02)
ax1.set_xlabel("Prefix length bin")
ax1.set_ylabel("Score")
ax1.set_title("Realtime prefix performance")
ax1.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "prefix_length_performance.png", dpi=160)
display(fig)
plt.close(fig)

prefix_curve


### 7.2 Confidence thresholds

Choose a threshold for each subtask using precision and support on the validation set. Save the thresholds in `label_map.json`.


In [ ]:
threshold_grid = np.round(np.arange(0.30, 0.91, 0.05), 2)
val_proba = subtask_model.predict_proba(X_val)
val_pred = np.argmax(val_proba, axis=1)
val_conf = val_proba.max(axis=1)
val_true = y_subtask[val_idx]

SUBTASK_CONFIDENCE_THRESHOLDS: dict[str, float] = {}
threshold_rows = []
for class_id, subtask_name in enumerate(subtask_encoder.classes_):
    predicted_mask = val_pred == class_id
    evaluated = []
    for candidate in threshold_grid:
        answered = predicted_mask & (val_conf >= candidate)
        support = int(answered.sum())
        if support < THRESHOLD_MIN_SUPPORT:
            continue
        precision = float((val_true[answered] == class_id).mean())
        coverage = float(support / max(1, int(predicted_mask.sum())))
        evaluated.append({"threshold": float(candidate), "precision": precision, "coverage": coverage, "support": support})

    meeting = [row for row in evaluated if row["precision"] >= PER_CLASS_PRECISION_TARGET]
    if meeting:
        chosen = meeting[0]
    elif evaluated:
        chosen = max(evaluated, key=lambda row: row["precision"])
    else:
        chosen = {"threshold": float(DEFAULT_CONFIDENCE_THRESHOLD), "precision": float("nan"), "coverage": float("nan"), "support": 0}

    SUBTASK_CONFIDENCE_THRESHOLDS[str(subtask_name)] = chosen["threshold"]
    threshold_rows.append({"subtask": str(subtask_name), "predicted_val_windows": int(predicted_mask.sum()), **chosen})

threshold_table = pd.DataFrame(threshold_rows)
threshold_table.to_csv(OUTPUT_DIR / "subtask_confidence_thresholds.csv", index=False)

applied_thresholds = np.array([SUBTASK_CONFIDENCE_THRESHOLDS[str(subtask_encoder.classes_[p])] for p in val_pred])
answered_mask = val_conf >= applied_thresholds
print(f"Validation with per-class thresholds: answers {answered_mask.mean():.1%} of windows")
if answered_mask.any():
    print(f"Accuracy when answering: {(val_pred[answered_mask] == val_true[answered_mask]).mean():.1%}")
if (~answered_mask).any():
    print(f"Accuracy on abstained windows (would have been): {(val_pred[~answered_mask] == val_true[~answered_mask]).mean():.1%}")
threshold_table

### 7.3 Reporting figures

These plots use saved evaluation tables, predictions, and model files. Figures are written to `FIGURE_DIR`.


In [ ]:
# Reporting figure 7.3.1: training-window class balance and per-subtask test F1.
from matplotlib.patches import Patch

rfcb_windows_path = TRAINING_INTERMEDIATE_DIR / "training_window_metadata.parquet"
rfcb_report_path = OUTPUT_DIR / "subtask_classification_report.csv"
if not (rfcb_windows_path.exists() and rfcb_report_path.exists()):
    print("Skipped: training-window metadata / classification report not found. Run sections 2-7 first.")
else:
    rfcb_family_palette = ["#4C78A8", "#F58518", "#E45756", "#72B7B2", "#54A24B", "#B279A2"]
    rfcb_family_colors = {
        rfcb_family: rfcb_family_palette[rfcb_index % len(rfcb_family_palette)]
        for rfcb_index, rfcb_family in enumerate(TASK_FAMILY_ORDER)
    }
    rfcb_counts = (
        pd.read_parquet(rfcb_windows_path, columns=["subtask"])["subtask"]
        .value_counts()
        .reindex(SUBTASK_ORDER)
        .fillna(0)
    )
    rfcb_report = pd.read_csv(rfcb_report_path, index_col=0)
    rfcb_rows = [rfcb_sub for rfcb_sub in SUBTASK_ORDER if rfcb_sub in rfcb_report.index]
    rfcb_f1 = rfcb_report.loc[rfcb_rows, "f1-score"].to_numpy(dtype=float)
    rfcb_support = rfcb_report.loc[rfcb_rows, "support"].to_numpy(dtype=float)
    rfcb_macro = float(rfcb_report.loc["macro avg", "f1-score"])
    rfcb_colors = [rfcb_family_colors[SUBTASK_INFO[rfcb_sub]["task_family"]] for rfcb_sub in rfcb_rows]
    rfcb_row_families = [SUBTASK_INFO[rfcb_sub]["task_family"] for rfcb_sub in rfcb_rows]
    rfcb_y = np.arange(len(rfcb_rows))

    fig, (rfcb_left, rfcb_right) = plt.subplots(
        1, 2, figsize=(13.5, 7.2), sharey=True, gridspec_kw={"wspace": 0.05}
    )
    rfcb_window_values = rfcb_counts.loc[rfcb_rows].to_numpy(dtype=float)
    rfcb_left.barh(rfcb_y, rfcb_window_values, color=rfcb_colors, alpha=0.92)
    rfcb_left.set_xscale("log")
    rfcb_left.set_xlim(right=rfcb_window_values.max() * 4)
    for rfcb_i, rfcb_value in enumerate(rfcb_window_values):
        rfcb_left.text(rfcb_value * 1.15, rfcb_i, f"{int(rfcb_value):,}", va="center", fontsize=7.5, color="#3F3F3F")
    rfcb_left.set_yticks(rfcb_y)
    rfcb_left.set_yticklabels(rfcb_rows, fontsize=9)
    rfcb_left.invert_yaxis()
    rfcb_left.set_xlabel("generated training windows (log scale)")
    rfcb_left.set_title("Class balance (all generated windows)", fontsize=11)

    rfcb_right.barh(rfcb_y, rfcb_f1, color=rfcb_colors, alpha=0.92)
    rfcb_right.set_xlim(0, 1.30)
    for rfcb_i, (rfcb_value, rfcb_n) in enumerate(zip(rfcb_f1, rfcb_support)):
        rfcb_right.text(rfcb_value + 0.015, rfcb_i, f"{rfcb_value:.2f}", va="center", fontsize=7.5, color="#3F3F3F")
        rfcb_right.text(1.14, rfcb_i, f"n={int(rfcb_n):,}", va="center", fontsize=7.2, color="#8A8A8A")
    rfcb_right.axvline(rfcb_macro, color="#555555", linestyle="--", linewidth=1.1)
    rfcb_right.text(rfcb_macro, -0.85, f"macro-F1 = {rfcb_macro:.2f}", ha="center", fontsize=8, color="#555555")
    rfcb_right.set_xlabel("F1 score on the held-out test split")
    rfcb_right.set_title("Per-subtask test F1 (n = test windows)", fontsize=11)

    for rfcb_axis in (rfcb_left, rfcb_right):
        for rfcb_i in range(1, len(rfcb_rows)):
            if rfcb_row_families[rfcb_i] != rfcb_row_families[rfcb_i - 1]:
                rfcb_axis.axhline(rfcb_i - 0.5, color="#B5B5B5", linewidth=0.8, linestyle=":")
    fig.legend(
        handles=[Patch(facecolor=rfcb_family_colors[rfcb_family], label=rfcb_family) for rfcb_family in TASK_FAMILY_ORDER],
        loc="lower center", ncol=3, fontsize=8, frameon=False,
    )
    fig.subplots_adjust(left=0.245, right=0.985, top=0.93, bottom=0.185, wspace=0.05)
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / "report_class_balance_f1.png", dpi=160, bbox_inches="tight")
    plt.show()

In [ ]:
# Reporting figure 7.3.2: where subtask errors go - same family vs different family.
from matplotlib.patches import Patch

rfef_pred_path = TRAINING_INTERMEDIATE_DIR / "subtask_test_predictions.parquet"
rfef_metrics_path = OUTPUT_DIR / "test_metrics.csv"
if not rfef_pred_path.exists():
    print("Skipped: subtask test predictions not found. Run sections 2-7 first.")
else:
    rfef_family_palette = ["#4C78A8", "#F58518", "#E45756", "#72B7B2", "#54A24B", "#B279A2"]
    rfef_family_colors = {
        rfef_family: rfef_family_palette[rfef_index % len(rfef_family_palette)]
        for rfef_index, rfef_family in enumerate(TASK_FAMILY_ORDER)
    }
    rfef_family_of = {rfef_sub: SUBTASK_INFO[rfef_sub]["task_family"] for rfef_sub in SUBTASK_ORDER}
    rfef_pred = pd.read_parquet(rfef_pred_path, columns=["subtask_true", "subtask_pred"])
    rfef_pred["outcome"] = np.select(
        [
            rfef_pred["subtask_pred"] == rfef_pred["subtask_true"],
            rfef_pred["subtask_pred"].map(rfef_family_of) == rfef_pred["subtask_true"].map(rfef_family_of),
        ],
        ["exact", "same_family"],
        default="cross_family",
    )
    rfef_table = (
        rfef_pred.groupby("subtask_true")["outcome"]
        .value_counts(normalize=True)
        .unstack(fill_value=0.0)
        .reindex(SUBTASK_ORDER)
        .fillna(0.0)
    )
    for rfef_column in ("exact", "same_family", "cross_family"):
        if rfef_column not in rfef_table.columns:
            rfef_table[rfef_column] = 0.0
    rfef_n = rfef_pred["subtask_true"].value_counts().reindex(SUBTASK_ORDER).fillna(0)
    rfef_exact_overall = float((rfef_pred["outcome"] == "exact").mean())
    rfef_family_overall = float((rfef_pred["outcome"] != "cross_family").mean())

    fig, ax = plt.subplots(figsize=(12, 7))
    rfef_y = np.arange(len(SUBTASK_ORDER))
    for rfef_i, rfef_sub in enumerate(SUBTASK_ORDER):
        rfef_color = rfef_family_colors[rfef_family_of[rfef_sub]]
        rfef_exact = float(rfef_table.loc[rfef_sub, "exact"])
        rfef_same = float(rfef_table.loc[rfef_sub, "same_family"])
        rfef_cross = float(rfef_table.loc[rfef_sub, "cross_family"])
        ax.barh(rfef_i, rfef_exact, color=rfef_color, alpha=0.95)
        ax.barh(rfef_i, rfef_same, left=rfef_exact, color=rfef_color, alpha=0.38)
        ax.barh(rfef_i, rfef_cross, left=rfef_exact + rfef_same, color="#C44E52", alpha=0.85)
        ax.text(1.02, rfef_i, f"n={int(rfef_n.loc[rfef_sub]):,}", va="center", fontsize=7.2, color="#8A8A8A")
    rfef_row_families = [rfef_family_of[rfef_sub] for rfef_sub in SUBTASK_ORDER]
    for rfef_i in range(1, len(SUBTASK_ORDER)):
        if rfef_row_families[rfef_i] != rfef_row_families[rfef_i - 1]:
            ax.axhline(rfef_i - 0.5, color="#B5B5B5", linewidth=0.8, linestyle=":")
    ax.set_yticks(rfef_y)
    ax.set_yticklabels(SUBTASK_ORDER, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, 1.14)
    ax.set_xlabel("share of test windows")
    ax.set_title(
        "Prediction outcome per true subtask\n"
        f"overall: exact {rfef_exact_overall:.1%}, correct family {rfef_family_overall:.1%} "
        "(subtask predictions mapped to families)",
        fontsize=11.5,
    )
    fig.legend(
        handles=[
            Patch(facecolor="#4F4F4F", label="exact subtask correct"),
            Patch(facecolor="#4F4F4F", alpha=0.35, label="wrong subtask, same family"),
            Patch(facecolor="#C44E52", alpha=0.85, label="wrong task family"),
        ],
        loc="lower center", ncol=3, fontsize=8, frameon=False,
    )
    fig.text(0.99, 0.005, "Solid/light shades follow the task-family colour of each row.",
             ha="right", fontsize=7.5, color="#8A8A8A")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / "report_within_family_error_split.png", dpi=160, bbox_inches="tight")
    plt.show()
    if rfef_metrics_path.exists():
        rfef_metrics = pd.read_csv(rfef_metrics_path)
        print("Cross-check vs test_metrics.csv (dedicated family head):")
        print(rfef_metrics.to_string(index=False))

In [ ]:
# Reporting figure 7.3.3: confidence calibration and the coverage/accuracy trade-off behind section 7.2.
rfcal_pred_path = TRAINING_INTERMEDIATE_DIR / "subtask_test_predictions.parquet"
if not rfcal_pred_path.exists():
    print("Skipped: subtask test predictions not found. Run sections 2-7 first.")
else:
    rfcal_pred = pd.read_parquet(rfcal_pred_path, columns=["subtask_confidence", "subtask_correct"])
    rfcal_confidence = rfcal_pred["subtask_confidence"].to_numpy(dtype=float)
    rfcal_correct = rfcal_pred["subtask_correct"].to_numpy(dtype=float)

    rfcal_bins = np.arange(0.0, 1.0001, 0.05)
    rfcal_indices = np.clip(np.digitize(rfcal_confidence, rfcal_bins) - 1, 0, len(rfcal_bins) - 2)
    rfcal_min_bin_windows = 100
    rfcal_centers, rfcal_accuracy, rfcal_bin_counts = [], [], []
    for rfcal_bin in range(len(rfcal_bins) - 1):
        rfcal_mask = rfcal_indices == rfcal_bin
        if rfcal_mask.sum() < rfcal_min_bin_windows:
            continue
        rfcal_centers.append(float(rfcal_confidence[rfcal_mask].mean()))
        rfcal_accuracy.append(float(rfcal_correct[rfcal_mask].mean()))
        rfcal_bin_counts.append(int(rfcal_mask.sum()))

    fig, (rfcal_left, rfcal_right) = plt.subplots(1, 2, figsize=(13, 5.4))
    rfcal_left.plot([0, 1], [0, 1], linestyle="--", color="#9D9D9D", linewidth=1, label="perfect calibration")
    rfcal_left.plot(rfcal_centers, rfcal_accuracy, marker="o", color="#4C78A8", label="empirical accuracy")
    rfcal_twin = rfcal_left.twinx()
    rfcal_twin.bar(rfcal_centers, rfcal_bin_counts, width=0.042, color="#9D9D9D", alpha=0.30)
    rfcal_twin.set_ylim(0, max(rfcal_bin_counts) * 3.4)
    rfcal_twin.set_ylabel("test windows per bin", fontsize=8, color="#8A8A8A")
    rfcal_twin.tick_params(axis="y", labelsize=7, colors="#8A8A8A")
    rfcal_left.set_xlim(0, 1)
    rfcal_left.set_ylim(0, 1.02)
    rfcal_left.set_xlabel("predicted confidence (top-1 subtask)")
    rfcal_left.set_ylabel("empirical accuracy")
    rfcal_left.set_title(f"Reliability of the subtask confidence (bins with <{rfcal_min_bin_windows} windows omitted)",
                         fontsize=11)
    rfcal_left.legend(loc="upper left", fontsize=8, frameon=False)

    rfcal_taus = np.round(np.arange(0.30, 0.9001, 0.05), 2)
    rfcal_coverage = [float((rfcal_confidence >= rfcal_tau).mean()) for rfcal_tau in rfcal_taus]
    rfcal_selective = [
        float(rfcal_correct[rfcal_confidence >= rfcal_tau].mean()) if (rfcal_confidence >= rfcal_tau).any() else np.nan
        for rfcal_tau in rfcal_taus
    ]
    rfcal_right.plot(rfcal_taus, rfcal_coverage, marker="o", color="#F58518", label="coverage (windows kept)")
    rfcal_right.plot(rfcal_taus, rfcal_selective, marker="s", color="#4C78A8", label="accuracy on kept windows")
    rfcal_right.axvline(DEFAULT_CONFIDENCE_THRESHOLD, color="#555555", linestyle="--", linewidth=1.1)
    rfcal_right.text(DEFAULT_CONFIDENCE_THRESHOLD + 0.008, 0.99, f"default τ = {DEFAULT_CONFIDENCE_THRESHOLD}",
                     va="top", fontsize=8, color="#555555")
    rfcal_right.set_xlabel("confidence threshold τ (abstain below)")
    rfcal_right.set_ylim(0, 1.02)
    rfcal_right.set_title("Coverage vs accuracy when abstaining below τ", fontsize=11)
    rfcal_right.legend(loc="lower left", fontsize=8, frameon=False)

    fig.tight_layout()
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / "report_confidence_calibration.png", dpi=160, bbox_inches="tight")
    plt.show()

In [ ]:
# Reporting figure 7.3.4: what the subtask model relies on - top XGBoost gain importances.
from matplotlib.patches import Patch

rffi_model_path = MODEL_DIR / "subtask_classifier.joblib"
rffi_vectorizer_path = MODEL_DIR / "feature_vectorizer.joblib"
if not (rffi_model_path.exists() and rffi_vectorizer_path.exists()):
    print("Skipped: deployed subtask classifier / vectorizer not found. Run section 9 (or a previous run) first.")
else:
    rffi_model = joblib.load(rffi_model_path)
    rffi_names = list(joblib.load(rffi_vectorizer_path).feature_names_)
    rffi_scores = rffi_model.get_booster().get_score(importance_type="gain")

    def rffi_resolve(rffi_key):
        if rffi_key.startswith("f") and rffi_key[1:].isdigit() and int(rffi_key[1:]) < len(rffi_names):
            return rffi_names[int(rffi_key[1:])]
        return rffi_key

    rffi_gains = {rffi_resolve(rffi_key): float(rffi_value) for rffi_key, rffi_value in rffi_scores.items()}
    rffi_total = sum(rffi_gains.values())
    rffi_top = sorted(rffi_gains.items(), key=lambda rffi_item: rffi_item[1], reverse=True)[:20]

    def rffi_group(rffi_name):
        if "=" in rffi_name:
            return rffi_name.split("=")[0]
        if ":" in rffi_name:
            return rffi_name.split(":")[0]
        return rffi_name

    rffi_group_meta = {
        "activity_count": ("#4C78A8", "activity counts (whole window)"),
        "activity_ratio": ("#9ECAE9", "activity shares (whole window)"),
        "recent_activity_count": ("#C6DBEF", "activity counts (recent tail)"),
        "family_count": ("#54A24B", "activity-family counts"),
        "family_ratio": ("#A1D99B", "activity-family shares"),
        "file_role_count": ("#72B7B2", "file-role counts"),
        "file_role_ratio": ("#B7E0DD", "file-role shares"),
        "bigram": ("#F58518", "activity 2-gram transitions"),
        "trigram": ("#FFBF79", "activity 3-gram transitions"),
        "first_activity": ("#B279A2", "first activity in window"),
        "last_activity": ("#83599C", "most recent activity"),
        "recent_command_category": ("#E45756", "recent command category"),
        "num": ("#7F7F7F", "numeric window statistics"),
    }

    rffi_labels = [rffi_name.replace("=", " = ") for rffi_name, _ in rffi_top]
    rffi_shares = [rffi_value / rffi_total for _, rffi_value in rffi_top]
    rffi_groups = [rffi_group(rffi_name) for rffi_name, _ in rffi_top]
    rffi_colors = [rffi_group_meta.get(rffi_g, ("#9D9D9D", rffi_g))[0] for rffi_g in rffi_groups]

    fig, ax = plt.subplots(figsize=(11.5, 7.6))
    rffi_y = np.arange(len(rffi_top))
    ax.barh(rffi_y, rffi_shares, color=rffi_colors, alpha=0.94)
    ax.set_yticks(rffi_y)
    ax.set_yticklabels(rffi_labels, fontsize=8.2, family="monospace")
    ax.invert_yaxis()
    ax.set_xlabel("gain importance (share of total gain)")
    ax.set_title("Top-20 features of the deployed subtask classifier", fontsize=12)
    rffi_present = []
    for rffi_g in rffi_groups:
        if rffi_g not in rffi_present:
            rffi_present.append(rffi_g)
    ax.legend(
        handles=[
            Patch(facecolor=rffi_group_meta.get(rffi_g, ("#9D9D9D", rffi_g))[0],
                  label=rffi_group_meta.get(rffi_g, ("#9D9D9D", rffi_g))[1])
            for rffi_g in rffi_present
        ],
        loc="lower right", fontsize=8, frameon=False, title="feature group", title_fontsize=8,
    )
    fig.tight_layout()
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIGURE_DIR / "report_feature_importance.png", dpi=160, bbox_inches="tight")
    plt.show()

## 8. Predict from an activity window

The example accepts activity tokens or events with context. It returns a subtask or a task family when the corresponding confidence threshold is met. Otherwise it reports uncertainty.


In [ ]:
SUBTASK_TO_FAMILY = {
    label.subtask: label.task_family
    for label in CLUSTER_TO_TASK.values()
}
SUBTASK_CLASS_FAMILIES = [SUBTASK_TO_FAMILY.get(str(name), "Unknown") for name in subtask_encoder.classes_]


def predict_task_from_events(events: list[dict[str, Any] | str], segment_context: dict[str, Any] | None = None) -> dict[str, Any]:
    features = extract_window_features(events, segment_context=segment_context)
    X = vectorizer.transform([features])
    proba = subtask_model.predict_proba(X)[0]
    order = np.argsort(proba)[::-1]
    subtask = str(subtask_encoder.inverse_transform([order[0]])[0])
    subtask_family = SUBTASK_TO_FAMILY.get(subtask, "Unknown")
    confidence = float(proba[order[0]])
    applied_threshold = float(SUBTASK_CONFIDENCE_THRESHOLDS.get(subtask, DEFAULT_CONFIDENCE_THRESHOLD))

    family_probabilities: dict[str, float] = {}
    for class_index, value in enumerate(proba):
        family_name = SUBTASK_CLASS_FAMILIES[class_index]
        family_probabilities[family_name] = family_probabilities.get(family_name, 0.0) + float(value)
    top_family, family_confidence = max(family_probabilities.items(), key=lambda item: item[1])

    if confidence >= applied_threshold:
        level = "subtask"
    elif family_confidence >= FAMILY_FALLBACK_THRESHOLD:
        level = "family"
    else:
        level = "uncertain"

    return {
        "level": level,
        "task_family": subtask_family if level == "subtask" else top_family,
        "subtask": subtask if level == "subtask" else None,
        "top_subtask": subtask,
        "confidence": confidence,
        "family_confidence": float(family_confidence),
        "applied_threshold": applied_threshold,
        "top3": [
            {
                "task_family": SUBTASK_TO_FAMILY.get(str(subtask_encoder.inverse_transform([idx])[0]), "Unknown"),
                "subtask": str(subtask_encoder.inverse_transform([idx])[0]),
                "confidence": float(proba[idx]),
            }
            for idx in order[:3]
        ],
        "used_events": len(events[-MAX_HISTORY_EVENTS:]),
    }


demo_sequences = {
    "completion-assisted editing": ["CODE_COMPLETION", "EDIT_BURST", "CODE_COMPLETION", "EDIT_BURST", "CODE_COMPLETION"],
    "edit-completion-navigation refinement": ["EDIT_BURST", "CODE_COMPLETION", "NAVIGATION_BURST", "CODE_COMPLETION", "EDIT_BURST"],
    "debug-guided edit repair": ["EDIT_BURST", "DEBUG_SESSION", "EDIT_BURST", "DEBUG_SESSION", "IDE_COMMAND_ACTIVITY"],
    "build feedback review": ["BUILD", "BUILD", "NAVIGATION_BURST", "EDIT_BURST", "BUILD"],
    "tight debugger interaction": ["DEBUG_SESSION", "DEBUG_SESSION", "IDE_COMMAND_ACTIVITY", "DEBUG_SESSION"],
    "vcs inspection/sync": ["VCS_ACTIVITY", "VCS_ACTIVITY", "VCS_ACTIVITY", "VCS_SYNC"],
    "context/file navigation": ["NAVIGATION_BURST", "IDE_COMMAND_ACTIVITY", "NAVIGATION_BURST"],
    "runtime-aware investigation": ["NAVIGATION_BURST", "DEBUG_SESSION", "NAVIGATION_BURST", "EDIT_BURST"],
    "command-driven debug interaction": ["IDE_COMMAND_ACTIVITY", "DEBUG_SESSION", "IDE_COMMAND_ACTIVITY", "DEBUG_SESSION"],
    "test-code editing (context events)": [
        {"activity": "EDIT_BURST", "file_role": "test"},
        {"activity": "NAVIGATION_BURST", "file_role": "test"},
        {"activity": "CODE_COMPLETION", "file_role": "test"},
        {"activity": "EDIT_BURST", "file_role": "test"},
    ],
    "symbol/usage investigation (context events)": [
        {"activity": "NAVIGATION_BURST", "navigation_kind": "find_usages", "file_role": "source"},
        {"activity": "IDE_COMMAND_ACTIVITY", "file_role": "source"},
        {"activity": "NAVIGATION_BURST", "navigation_kind": "find_usages", "file_role": "source"},
        {"activity": "EDIT_BURST", "file_role": "source"},
    ],
}

demo_rows = []
for name, sequence in demo_sequences.items():
    prediction = predict_task_from_events(sequence)
    demo_rows.append(
        {
            "demo": name,
            "events": " -> ".join(event["activity"] if isinstance(event, dict) else str(event) for event in sequence),
            "level": prediction["level"],
            "task_family": prediction["task_family"],
            "top_subtask": prediction["top_subtask"],
            "confidence": prediction["confidence"],
            "family_confidence": prediction["family_confidence"],
            "applied_threshold": prediction["applied_threshold"],
        }
    )

demo_frame = pd.DataFrame(demo_rows)
demo_frame.to_csv(OUTPUT_DIR / "runtime_demo_predictions.csv", index=False)
demo_frame

## 9. Save the model files

Save the classifiers, vectorizer, label map, feature schema, and training metadata.


In [ ]:
joblib.dump(vectorizer, MODEL_DIR / "feature_vectorizer.joblib")
joblib.dump(subtask_model, MODEL_DIR / "subtask_classifier.joblib")
joblib.dump(family_model, MODEL_DIR / "task_family_classifier.joblib")

label_map = {
    "schema_version": f"task_classifier_v4::{TAXONOMY['schema_version']}::{TAXONOMY['run_id']}",
    "task_families": list(family_encoder.classes_),
    "subtasks": list(subtask_encoder.classes_),
    "subtask_to_family": SUBTASK_TO_FAMILY,
    "cluster_to_task": {
        str(cluster_id): {
            "task_family": label.task_family,
            "subtask": label.subtask,
            "description": label.description,
        }
        for cluster_id, label in sorted(CLUSTER_TO_TASK.items())
    },
    "default_confidence_threshold": DEFAULT_CONFIDENCE_THRESHOLD,
    "subtask_confidence_thresholds": SUBTASK_CONFIDENCE_THRESHOLDS,
    "family_fallback_threshold": FAMILY_FALLBACK_THRESHOLD,
    "per_class_precision_target": PER_CLASS_PRECISION_TARGET,
}

feature_schema = {
    "schema_version": "intermediate_event_window_features_v1",
    "intermediate_event_fields": [
        "timestamp",
        "activity",
        "action_type",
        "family",
        "command_category",
        "navigation_kind",
        "debug_action",
        "vcs_action",
        "file_role",
        "source_event",
        "source_action",
    ],
    "max_history_events": MAX_HISTORY_EVENTS,
    "uses_segment_context": False,
    "uses_context_trace": True,
    "numeric_context_columns": NUMERIC_CONTEXT_COLUMNS,
    "categorical_context_columns": CATEGORICAL_CONTEXT_COLUMNS,
    "feature_count": len(feature_names),
    "feature_names": feature_names,
}

metadata = {
    "schema_version": "task_classifier_training_metadata_v1",
    "created_from_notebook": "03_training_classifier.ipynb",
    "phase2_outputs": {
        "segments_path": str(SEGMENTS_PATH),
        "cluster_summary_path": str(CLUSTER_SUMMARY_PATH),
        "pipeline_summary_path": str(PIPELINE_SUMMARY_PATH),
        "pipeline_summary": pipeline_summary,
    },
    "training_config": {
        "min_cluster_probability": MIN_CLUSTER_PROBABILITY,
        "min_prefix_events": MIN_PREFIX_EVENTS,
        "prefix_samples_per_segment": PREFIX_SAMPLES_PER_SEGMENT,
        "max_history_events": MAX_HISTORY_EVENTS,
        "split_strategy": split_strategy,
        "prefer_xgboost": PREFER_XGBOOST,
        "use_gpu_if_available": USE_GPU_IF_AVAILABLE,
        "segment_context_features": False,
        "class_balanced_sample_weights": True,
    },
    "data_summary": {
        "source_segments": int(len(segments)),
        "labeled_segments": int(len(labeled_segments)),
        "training_examples": int(len(example_meta)),
        "train_examples": int(len(train_idx)),
        "validation_examples": int(len(val_idx)),
        "test_examples": int(len(test_idx)),
        "task_labels": int(example_meta["task_label"].nunique()),
    },
    "subtask_model": {k: v for k, v in subtask_training["best"].items() if k != "estimator"},
    "family_model": {k: v for k, v in family_training["best"].items() if k != "estimator"},
    "test_metrics": metrics_frame.to_dict(orient="records"),
}

(MODEL_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2, ensure_ascii=False), encoding="utf-8")
(MODEL_DIR / "feature_schema.json").write_text(json.dumps(feature_schema, indent=2, ensure_ascii=False), encoding="utf-8")
(MODEL_DIR / "training_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

segment_label_summary.to_csv(OUTPUT_DIR / "segment_label_summary.csv", index=False)
example_meta.to_parquet(TRAINING_INTERMEDIATE_DIR / "training_window_metadata.parquet", index=False)

print("Saved artifacts to:", MODEL_DIR)
print("Saved reports to:", OUTPUT_DIR)
print("Primary metric summary:")
metrics_frame


## 10. Use the trained model

Extract features from a recent activity window, transform them with the saved vectorizer, and pass them to the classifier. The label map gives the class names, task families, and confidence thresholds.
